In [0]:
# Celda 1: instalar dependencia y reiniciar Python (siempre primero)
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
# Celda 2: recién aquí se retoma el código normal
from pyspark.sql import functions as F

df = spark.table("wine_quality_raw")

# Variable objetivo binaria: calidad buena (>=7) vs no buena
df = df.withColumn("target", F.when(F.col("quality") >= 7, 1).otherwise(0))

# id único requerido por Feature Store
df = df.withColumn("wine_id", F.monotonically_increasing_id())

feature_cols = [c for c in df.columns if c not in ["quality", "target", "wine_id"]]
df_features = df.select(["wine_id"] + feature_cols)
df_labels = df.select("wine_id", "target")

df_features.write.format("delta").mode("overwrite").saveAsTable("wine_quality_features_raw")
df_labels.write.format("delta").mode("overwrite").saveAsTable("wine_quality_labels")

In [0]:
# Celda 3: registrar en Feature Store
from databricks.feature_engineering import FeatureEngineeringClient
fe = FeatureEngineeringClient()

fe.create_table(
    name="wine_quality_features",
    primary_keys=["wine_id"],
    df=df_features,
    description="Variables fisicoquímicas del vino para predicción de calidad"
)

print("Feature table 'wine_quality_features' registrada correctamente.")